In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 500
num_features = 10

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0,    # X5 effect
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 1

#Y_raw = X @ beta_true + noise  + X[:,4:5]*X[:,4:5] +X[:,2:3]*X[:,3:4]
Y_raw = X @ beta_true + noise  +X[:,2:3]*X[:,3:4]+ X[:,4:5]*X[:,4:5]

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 1.9074e+00],
        [ 1.9847e+00],
        [-1.4741e+00],
        [ 6.7115e-01],
        [-1.6332e-01],
        [ 2.9981e+00],
        [-1.9934e-01],
        [ 9.8232e-02],
        [ 5.5160e-02],
        [ 3.1714e-02],
        [-1.4717e-03]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([12, 12])

Attention Matrix:
 tensor([[9.4685e-02, 8.4928e-02, 5.3336e-02, 5.4023e-02, 1.1288e-01, 7.3003e-02,
         1.4410e-01, 7.0847e-02, 1.4272e-01, 6.8323e-02, 4.4409e-02, 5.6748e-02],
        [8.8187e-02, 5.6457e-02, 8.9850e-02, 1.2643e-01, 6.3798e-02, 9.5291e-02,
         6.8500e-02, 6.7422e-02, 9.3100e-02, 1.0739e-01, 6.6795e-02, 7.6784e-02],
        [5.0870e-02, 6.8003e-02, 7.6568e-02, 6.1979e-02, 6.6270e-02, 9.6725e-02,
         3.9581e-02, 4.6671e-02, 7.4258e-02, 8.9704e-02, 7.3824e-02, 2.5555e-01],
        [4.4398e-02, 3.7216e-02, 2.2194e-02, 4.6632e-02, 5.8179e-02, 6.6058e-02,
         6.1216e-02, 7.6153e-02, 3.8693e-02, 4.4773e-02, 7.4496e-02, 4.2999e-01],
        [7.7713e-02, 5.4981e-02, 5.4560e-02, 4.1956e-02, 6.6414e-02, 4.6844e-02,
         5.8469e-02, 3.5440e-02, 5.3324e-02, 3.6995e-02, 4.6100e-02, 4.2721e-01],
        [9.6358e-02, 8.8274e-02, 1.1677e-01, 6.1779e-02, 1.0986e-01, 5.6178e-02,
         6.3569e-02, 6.2816e-02, 6.6097

In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

#A = scores[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([11, 11])

Attention Matrix(拿掉Y):
 tensor([[0.0947, 0.0849, 0.0533, 0.0540, 0.1129, 0.0730, 0.1441, 0.0708, 0.1427,
         0.0683, 0.0444],
        [0.0882, 0.0565, 0.0898, 0.1264, 0.0638, 0.0953, 0.0685, 0.0674, 0.0931,
         0.1074, 0.0668],
        [0.0509, 0.0680, 0.0766, 0.0620, 0.0663, 0.0967, 0.0396, 0.0467, 0.0743,
         0.0897, 0.0738],
        [0.0444, 0.0372, 0.0222, 0.0466, 0.0582, 0.0661, 0.0612, 0.0762, 0.0387,
         0.0448, 0.0745],
        [0.0777, 0.0550, 0.0546, 0.0420, 0.0664, 0.0468, 0.0585, 0.0354, 0.0533,
         0.0370, 0.0461],
        [0.0964, 0.0883, 0.1168, 0.0618, 0.1099, 0.0562, 0.0636, 0.0628, 0.0661,
         0.1119, 0.1574],
        [0.0573, 0.0922, 0.0686, 0.0636, 0.0653, 0.0941, 0.0824, 0.0894, 0.0476,
         0.0841, 0.0578],
        [0.0463, 0.0594, 0.0287, 0.0281, 0.0509, 0.0433, 0.0489, 0.0414, 0.0594,
         0.0245, 0.0399],
        [0.0885, 0.0854, 0.0952, 0.0631, 0.0812, 0.0720, 0.0978, 0.08

In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[1.6289, 1.4610, 0.9176, 0.9294, 1.9419, 1.2559, 2.4789, 1.2188, 2.4552,
         1.1754, 0.7640],
        [1.5171, 0.9712, 1.5457, 2.1750, 1.0975, 1.6393, 1.1784, 1.1599, 1.6016,
         1.8474, 1.1491],
        [0.8751, 1.1699, 1.3172, 1.0662, 1.1401, 1.6640, 0.6809, 0.8029, 1.2775,
         1.5432, 1.2700],
        [0.7638, 0.6402, 0.3818, 0.8022, 1.0009, 1.1364, 1.0531, 1.3101, 0.6656,
         0.7702, 1.2816],
        [1.3369, 0.9458, 0.9386, 0.7218, 1.1425, 0.8059, 1.0059, 0.6097, 0.9173,
         0.6364, 0.7931],
        [1.6577, 1.5186, 2.0087, 1.0628, 1.8900, 0.9664, 1.0936, 1.0806, 1.1371,
         1.9250, 2.7084],
        [0.9858, 1.5859, 1.1807, 1.0936, 1.1240, 1.6195, 1.4174, 1.5380, 0.8196,
         1.4473, 0.9935],
        [0.7962, 1.0220, 0.4939, 0.4835, 0.8761, 0.7448, 0.8415, 0.7121, 1.0220,
         0.4210, 0.6868],
        [1.5218, 1.4695, 1.6381, 1.0864, 1.3962, 1.2380, 1.6829, 1.5230, 1.0932,
         1.6189, 2.0082],
        [1.0219, 1.4620, 1.6642, 1.64

In [11]:
X_train.T @ X_train

tensor([[400.0000,   8.2436,  40.9077,  37.6154,   6.2691,  12.9105,  24.0930,
         -31.6658,  -1.2076, -24.8618,   8.0275],
        [  8.2436, 367.4192,  51.2420,  12.8773,  26.1457,   6.0824,  13.8111,
          13.6067,  25.3908, -20.9918, -12.6170],
        [ 40.9077,  51.2420, 361.0372, -28.9374,  -6.7591,   5.9214,  18.6886,
          -6.4543,   3.1067,  11.2943, -19.6148],
        [ 37.6154,  12.8773, -28.9374, 396.2118,  30.7690,  13.4805, -23.3120,
           5.3517,   4.1780, -16.6970,  -8.4535],
        [  6.2691,  26.1457,  -6.7591,  30.7690, 412.1182,   3.5399, -26.5479,
          10.2705, -30.1646, -22.0729, -26.5194],
        [ 12.9105,   6.0824,   5.9214,  13.4805,   3.5399, 371.5383,  -0.5520,
          14.4429,   9.4724, -17.9609,  39.8122],
        [ 24.0930,  13.8111,  18.6886, -23.3120, -26.5479,  -0.5520, 391.7788,
           3.5460,  12.8999, -26.2938,  -3.3090],
        [-31.6658,  13.6067,  -6.4543,   5.3517,  10.2705,  14.4429,   3.5460,
         393.3328,

In [12]:

if torch.linalg.det(A)<0:
    beta_attention = torch.linalg.solve(X_train.T @ X_train + A.T@A, X_train.T @ Y_train)
else:
    beta_attention = torch.linalg.solve(X_train.T @ X_train - A.T@A, X_train.T @ Y_train)




In [13]:
torch.linalg.det(A)

tensor(-3.3670e-15, grad_fn=<LinalgDetBackward0>)

In [14]:
torch.linalg.det(A.T@A)

tensor(1.1334e-29, grad_fn=<LinalgDetBackward0>)

In [15]:
torch.linalg.det(X_train.T @ X_train)

tensor(2.2694e+28)

In [16]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

In [17]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 1.9533],
        [ 1.9770],
        [-1.5217],
        [ 0.7117],
        [-0.1905],
        [ 2.9768],
        [-0.1980],
        [ 0.1589],
        [ 0.0369],
        [ 0.0481],
        [ 0.0392]])


In [18]:
print(
    "Attention MSE:",
    mse_attention.item()
)

print()

print(
    "OLS MSE:      ",
    mse_ols.item()
)

Attention MSE: 2.8126659393310547

OLS MSE:       2.8133840560913086
